In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from slum_segmentation import helper_functions
from slum_segmentation import np_dataset
from sklearn.model_selection import train_test_split

# Check amount of data per city compared to total

In [3]:
total_pixels = 0

for city in vars(np_dataset.cities).values():
    image = city.get_image()
    image_pixels = image.shape[1] * image.shape[2]
    total_pixels += image_pixels

for city in vars(np_dataset.cities).values():
    image = city.get_image()
    image_pixels = image.shape[1] * image.shape[2]
    print("name:",city.name)
    print("shape:",image.shape)
    print(f"% of dataset: {round(100 * image_pixels/total_pixels, 2)}%")
    print()

name: capetownsmall
shape: (10, 1479, 2652)
% of dataset: 28.66%

name: eldaein
shape: (10, 1111, 943)
% of dataset: 7.66%

name: elgeneina
shape: (10, 882, 899)
% of dataset: 5.79%

name: kianda
shape: (10, 42, 34)
% of dataset: 0.01%

name: kibera
shape: (10, 118, 256)
% of dataset: 0.22%

name: lower
shape: (10, 44, 237)
% of dataset: 0.08%

name: makoko
shape: (10, 149, 269)
% of dataset: 0.29%

name: medellin
shape: (10, 101, 143)
% of dataset: 0.11%

name: mumbai
shape: (10, 3927, 1993)
% of dataset: 57.19%



# Convert to pixel-wise format for machine learning, and make train/val/test splits for each city.
# Distribution:
# Training set: `70%`
# Validation set: `15%`
# Testing set: `15%`

In [4]:
x_train_cities = []
y_train_cities = []

x_val_cities = []
y_val_cities = []

x_test_cities = []
y_test_cities = []

for city in vars(np_dataset.cities).values():
    image = city.get_image()
    mask = city.get_mask()

    city_x = image.reshape(10, -1).T
    city_y = mask.reshape(-1)

    x_train, x_other, y_train, y_other = train_test_split(
        city_x,
        city_y,
        test_size=0.3,
        random_state=42,
        stratify=city_y
    )

    x_val, x_test, y_val, y_test = train_test_split(
        x_other,
        y_other,
        test_size=0.5,
        random_state=42,
        stratify=y_other
    )

    x_train_cities.append(x_train)
    y_train_cities.append(y_train)

    x_val_cities.append(x_val)
    y_val_cities.append(y_val)

    x_test_cities.append(x_test)
    y_test_cities.append(y_test)

x_train_combined = np.concatenate(x_train_cities, axis=0)
y_train_combined = np.concatenate(y_train_cities, axis=0)

x_val_combined = np.concatenate(x_val_cities, axis=0)
y_val_combined = np.concatenate(y_val_cities, axis=0)

x_test_combined = np.concatenate(x_test_cities, axis=0)
y_test_combined = np.concatenate(y_test_cities, axis=0)

print(x_train_combined.shape, y_train_combined.shape)
print(x_val_combined.shape, y_val_combined.shape)
print(x_test_combined.shape, y_test_combined.shape)

(9580194, 10) (9580194,)
(2052899, 10) (2052899,)
(2052905, 10) (2052905,)


# Perform percentile normalization on training set. Percentiles `1` and `99` will be used. Test set will be normalized according to the percentile values obtained from training set to prevent data leakage.

In [6]:
p1 = np.percentile(x_train_combined, 1, axis=0)
p99 = np.percentile(x_train_combined, 99, axis=0)

x_train_combined_norm = (x_train_combined - p1) / (p99 - p1)
x_val_combined_norm = (x_val_combined - p1) / (p99 - p1)
x_test_combined_norm = (x_test_combined - p1) / (p99 - p1)

x_train_combined_norm = np.clip(x_train_combined_norm, 0, 1)
x_val_combined_norm = np.clip(x_val_combined_norm, 0, 1)
x_test_combined_norm = np.clip(x_test_combined_norm, 0, 1)

# Save splits

In [7]:
np.save(f"../data/machine_learning_splits/p1.npy", p1)
np.save(f"../data/machine_learning_splits/p99.npy", p99)

np.save(f"../data/machine_learning_splits/combined/x_train.npy", x_train_combined_norm)
np.save(f"../data/machine_learning_splits/combined/y_train.npy", y_train_combined)

np.save(f"../data/machine_learning_splits/combined/x_val.npy", x_val_combined_norm)
np.save(f"../data/machine_learning_splits/combined/y_val.npy", y_val_combined)

np.save(f"../data/machine_learning_splits/combined/x_test.npy", x_test_combined_norm)
np.save(f"../data/machine_learning_splits/combined/y_test.npy", y_test_combined)

for city in vars(np_dataset.cities).values():
    image = city.get_image()
    mask = city.get_mask()

    city_x = image.reshape(10, -1).T
    city_y = mask.reshape(-1)

    x_train, x_other, y_train, y_other = train_test_split(
        city_x,
        city_y,
        test_size=0.3,
        random_state=42,
        stratify=city_y
    )

    x_val, x_test, y_val, y_test = train_test_split(
        x_other,
        y_other,
        test_size=0.5,
        random_state=42,
        stratify=y_other
    )

    x_train_norm = (x_train - p1) / (p99 - p1)
    x_val_norm = (x_val - p1) / (p99 - p1)
    x_test_norm = (x_test - p1) / (p99 - p1)

    x_train_norm = np.clip(x_train_norm, 0, 1)
    x_val_norm = np.clip(x_val_norm, 0, 1)
    x_test_norm = np.clip(x_test_norm, 0, 1)

    np.save(f"../data/machine_learning_splits/{city.name}/x_train.npy", x_train_norm)
    np.save(f"../data/machine_learning_splits/{city.name}/y_train.npy", y_train)

    np.save(f"../data/machine_learning_splits/{city.name}/x_val.npy", x_val_norm)
    np.save(f"../data/machine_learning_splits/{city.name}/y_val.npy", y_val)

    np.save(f"../data/machine_learning_splits/{city.name}/x_test.npy", x_test_norm)
    np.save(f"../data/machine_learning_splits/{city.name}/y_test.npy", y_test)

# Verify Changes

In [8]:
from slum_segmentation import ml_dataset

for city in vars(ml_dataset.cities).values():
    x_train, y_train = city.get_train_data()
    x_val, y_val = city.get_val_data()
    x_test, y_test = city.get_test_data()

    print(city.name)
    print("training:")
    print(f"x:, {x_train.shape}, {x_train.dtype}, min:{x_train.min()}, max:{x_train.max()}")
    print(f"y:, {y_train.shape}, {y_train.dtype}, min:{y_train.min()}, max:{y_train.max()}")
    print("validation:")
    print(f"x:, {x_val.shape}, {x_val.dtype}, min:{x_val.min()}, max:{x_val.max()}")
    print(f"y:, {y_val.shape}, {y_val.dtype}, min:{y_val.min()}, max:{y_val.max()}")
    print("testing:")
    print(f"x:, {x_test.shape}, {x_test.dtype}, min:{x_test.min()}, max:{x_test.max()}")
    print(f"y:, {y_test.shape}, {y_test.dtype}, min:{y_test.min()}, max:{y_test.max()}")
    print()

combined
training:
x:, (9580194, 10), float32, min:0.0, max:1.0
y:, (9580194,), uint8, min:0, max:1
validation:
x:, (2052899, 10), float32, min:0.0, max:1.0
y:, (2052899,), uint8, min:0, max:1
testing:
x:, (2052905, 10), float32, min:0.0, max:1.0
y:, (2052905,), uint8, min:0, max:1

capetownsmall
training:
x:, (2745615, 10), float32, min:0.0, max:1.0
y:, (2745615,), uint8, min:0, max:1
validation:
x:, (588346, 10), float32, min:0.0, max:1.0
y:, (588346,), uint8, min:0, max:1
testing:
x:, (588347, 10), float32, min:0.0, max:1.0
y:, (588347,), uint8, min:0, max:1

eldaein
training:
x:, (733371, 10), float32, min:0.0, max:1.0
y:, (733371,), uint8, min:0, max:1
validation:
x:, (157151, 10), float32, min:0.0, max:1.0
y:, (157151,), uint8, min:0, max:1
testing:
x:, (157151, 10), float32, min:0.0, max:1.0
y:, (157151,), uint8, min:0, max:1

elgeneina
training:
x:, (555042, 10), float32, min:0.15012942254543304, max:1.0
y:, (555042,), uint8, min:0, max:1
validation:
x:, (118938, 10), float32, 